## Load
Load the synthetic fishing-trip data and establish reproducible modeling settings.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.inspection import permutation_importance, PartialDependenceDisplay
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import roc_auc_score, log_loss, accuracy_score, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
import joblib

RANDOM_STATE = 355
DATA_PATH = Path("data") / "fishing_trips.csv"
df = pd.read_csv(DATA_PATH)
print(df.shape)

## Check
Check schema quality, target rates, duplicates, and categorical value counts before modeling.

In [1]:
print(df.dtypes)
print("missing values", int(df.isna().sum().sum()))
print("duplicate trip_id", int(df.trip_id.duplicated().sum()))
print("overall base rate", float(df.productive.mean()))
print("by trip_type\n", df.groupby("trip_type").productive.agg(["count", "mean"]))
print("by season\n", df.groupby("season").productive.agg(["count", "mean"]))
for col in ["trip_type", "species_targeted", "tide_stage_start", "month"]:
    print(f"{col} counts\n", df[col].value_counts().sort_index())

NameError: name 'df' is not defined

## Split
Define the prediction-time feature contract and split by season so future season 3 remains untouched until final evaluation.

In [ ]:
FEATURES = ["trip_type", "trip_hours", "month", "species_targeted", "party_size", "tide_stage_start", "moon_phase", "days_since_front", "fcst_wind_mph", "fcst_wave_height_ft", "fcst_air_temp_f", "water_temp_f", "trailing_3day_catch_rate"]
EXCLUDED = {"bites_logged_during_trip": "post-trip leakage", "trip_id": "row identifier", "trip_date": "used for time context, not a feature", "season": "used for time split, not a feature"}
CATEGORICAL = ["trip_type", "species_targeted", "tide_stage_start"]
NUMERIC = [c for c in FEATURES if c not in CATEGORICAL]
train = df[df.season.isin([1, 2])].copy()
test = df[df.season == 3].copy()
train1 = df[df.season == 1].copy()
val2 = df[df.season == 2].copy()
print("train", train.shape, "base rate", train.productive.mean())
print("test", test.shape, "base rate", test.productive.mean())

## Baselines
Fit the majority and month-rule baselines using training seasons only.

In [ ]:
majority_value = int(train.productive.mean() >= 0.5)
month_rates = train.groupby("month").productive.mean()
def majority_score(frame): return np.full(len(frame), majority_value, dtype=float)
def month_score(frame): return frame.month.map(month_rates).fillna(train.productive.mean()).gt(0.5).astype(float).to_numpy()

## Models and comparison
Compare logistic regression and gradient boosting on season 2 after fitting only on season 1, then refit the winner on seasons 1 and 2.

In [ ]:
def make_preprocessor():
    return ColumnTransformer([
        ("categorical", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CATEGORICAL),
        ("numeric", StandardScaler(), NUMERIC),
    ])
def make_logistic():
    return Pipeline([("preprocess", make_preprocessor()), ("model", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE))])
def make_boosting():
    return Pipeline([("preprocess", make_preprocessor()), ("model", GradientBoostingClassifier(random_state=RANDOM_STATE))])
def model_scores(y, prob, threshold=0.5):
    return {"auc": float(roc_auc_score(y, prob)), "log_loss": float(log_loss(y, prob, labels=[0,1])), "accuracy": float(accuracy_score(y, prob >= threshold))}
X1, y1 = train1[FEATURES], train1.productive
X2, y2 = val2[FEATURES], val2.productive
logit_validation = make_logistic().fit(X1, y1)
gb_validation = make_boosting().fit(X1, y1)
comparison = pd.DataFrame({
    "majority": model_scores(y2, majority_score(val2)),
    "month_rule": model_scores(y2, month_score(val2)),
    "logistic_regression": model_scores(y2, logit_validation.predict_proba(X2)[:,1]),
    "gradient_boosting": model_scores(y2, gb_validation.predict_proba(X2)[:,1]),
}).T
print(comparison)
chosen_name = "gradient_boosting" if comparison.loc["gradient_boosting", "auc"] > comparison.loc["logistic_regression", "auc"] else "logistic_regression"
print("Chosen model:", chosen_name)
chosen_model = make_boosting() if chosen_name == "gradient_boosting" else make_logistic()
chosen_model.fit(train[FEATURES], train.productive)

## Costs and threshold
Set the captain-facing mistake costs and derive the decision threshold rather than hard-coding it.

In [ ]:
COST_RUN_UNPRODUCTIVE = 300.0
COST_RESCHEDULE_PRODUCTIVE = 600.0
THRESHOLD = COST_RUN_UNPRODUCTIVE / (COST_RUN_UNPRODUCTIVE + COST_RESCHEDULE_PRODUCTIVE)
print("threshold", THRESHOLD)
def cost_summary(y, prob, threshold):
    pred = (np.asarray(prob) >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0,1]).ravel()
    total = fp * COST_RUN_UNPRODUCTIVE + fn * COST_RESCHEDULE_PRODUCTIVE
    return {"tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp), "total_cost": float(total), "cost_per_trip": float(total / len(y))}

## Test evaluation
Evaluate season 3 once at the cost threshold, compare both baselines, and inspect costs across thresholds.

In [ ]:
y_test = test.productive.to_numpy()
model_prob = chosen_model.predict_proba(test[FEATURES])[:,1]
test_metrics = {"auc": float(roc_auc_score(y_test, model_prob)), "accuracy": float(accuracy_score(y_test, model_prob >= THRESHOLD)), **cost_summary(y_test, model_prob, THRESHOLD)}
baseline_test = {
    "majority": {"auc": float(roc_auc_score(y_test, majority_score(test))), "accuracy": float(accuracy_score(y_test, majority_score(test))), **cost_summary(y_test, majority_score(test), THRESHOLD)},
    "month_rule": {"auc": float(roc_auc_score(y_test, month_score(test))), "accuracy": float(accuracy_score(y_test, month_score(test))), **cost_summary(y_test, month_score(test), THRESHOLD)},
}
threshold_sweep = []
for threshold in np.arange(0.05, 0.951, 0.05):
    row = cost_summary(y_test, model_prob, float(threshold)); row["threshold"] = round(float(threshold), 2); threshold_sweep.append(row)
threshold_sweep = pd.DataFrame(threshold_sweep)
print("model", test_metrics)
print("baselines", baseline_test)
print(threshold_sweep.to_string(index=False))

## Segment analysis
Measure AUC and accuracy by trip type, species, and month, and identify the weakest segment by AUC.

In [ ]:
segment_metrics = {}
for dimension in ["trip_type", "species_targeted", "month"]:
    segment_metrics[dimension] = {}
    for value, group in test.groupby(dimension, sort=True):
        idx = group.index - test.index.min()
        probs = model_prob[test.index.get_indexer(group.index)]
        segment_metrics[dimension][str(value)] = {"n": int(len(group)), "auc": float(roc_auc_score(group.productive, probs)) if group.productive.nunique() > 1 else None, "accuracy": float(accuracy_score(group.productive, probs >= THRESHOLD))}
valid_segments = [(dim, val, m["auc"]) for dim, vals in segment_metrics.items() for val, m in vals.items() if m["auc"] is not None]
weakest = min(valid_segments, key=lambda x: x[2])
print("weakest segment", weakest)

## Save artifacts
Save the fitted pipeline and a JSON metadata contract consumed by the app and documentation.

In [ ]:
numeric_ranges = {c: {"min": float(train[c].min()), "max": float(train[c].max())} for c in NUMERIC}
allowed_categories = {c: sorted(train[c].dropna().unique().tolist()) for c in CATEGORICAL}
allowed_species_by_trip_type = {k: sorted(v.species_targeted.unique().tolist()) for k, v in train.groupby("trip_type")}
allowed_months_by_trip_type = {k: sorted(v.month.unique().astype(int).tolist()) for k, v in train.groupby("trip_type")}
meta = {"synthetic_data_notice": "Synthetic data: this shows the pipeline works, not that real fish behave this way.", "features": FEATURES, "excluded": EXCLUDED, "costs": {"run_unproductive": COST_RUN_UNPRODUCTIVE, "reschedule_productive": COST_RESCHEDULE_PRODUCTIVE}, "threshold": THRESHOLD, "chosen_model": chosen_name, "test": test_metrics, "baselines": baseline_test, "segments": segment_metrics, "weakest_segment": {"dimension": weakest[0], "value": weakest[1], "auc": weakest[2]}, "training_numeric_ranges": numeric_ranges, "allowed_categories": allowed_categories, "allowed_species_by_trip_type": allowed_species_by_trip_type, "allowed_months_by_trip_type": allowed_months_by_trip_type, "train_rows": int(len(train)), "test_rows": int(len(test)), "test_season": 3, "base_rate_overall": float(df.productive.mean()), "base_rates_by_trip_type": {k: float(v) for k,v in df.groupby("trip_type").productive.mean().items()}, "base_rates_by_season": {str(k): float(v) for k,v in df.groupby("season").productive.mean().items()}, "threshold_sweep": threshold_sweep.to_dict(orient="records")}
joblib.dump(chosen_model, "model.joblib")
Path("model_meta.json").write_text(json.dumps(meta, indent=2), encoding="utf-8")

## Verification
Run leakage, temporal-feature, split-sensitivity, traceability, and planted-effect recovery checks without changing the saved production model.

In [ ]:
leaky_features = FEATURES + ["bites_logged_during_trip"]
leaky_cat = CATEGORICAL
leaky_num = [c for c in leaky_features if c not in leaky_cat]
leaky_pre = ColumnTransformer([("categorical", OneHotEncoder(handle_unknown="ignore", sparse_output=False), leaky_cat), ("numeric", StandardScaler(), leaky_num)])
leaky_model = Pipeline([("preprocess", leaky_pre), ("model", GradientBoostingClassifier(random_state=RANDOM_STATE))])
leaky_model.fit(train[leaky_features], train.productive)
leaky_auc = float(roc_auc_score(y_test, leaky_model.predict_proba(test[leaky_features])[:,1]))
without_trailing = [c for c in FEATURES if c != "trailing_3day_catch_rate"]
no_trailing_cat = [c for c in CATEGORICAL if c in without_trailing]
no_trailing_num = [c for c in without_trailing if c not in no_trailing_cat]
no_trailing_pre = ColumnTransformer([("categorical", OneHotEncoder(handle_unknown="ignore", sparse_output=False), no_trailing_cat), ("numeric", StandardScaler(), no_trailing_num)])
no_trailing_model = Pipeline([("preprocess", no_trailing_pre), ("model", GradientBoostingClassifier(random_state=RANDOM_STATE))]).fit(train[without_trailing], train.productive)
no_trailing_auc = float(roc_auc_score(y_test, no_trailing_model.predict_proba(test[without_trailing])[:,1]))
random_train, random_test = train_test_split(df, test_size=len(test), random_state=RANDOM_STATE, stratify=df.productive)
random_model = make_boosting().fit(random_train[FEATURES], random_train.productive)
random_auc = float(roc_auc_score(random_test.productive, random_model.predict_proba(random_test[FEATURES])[:,1]))
# Rank-based AUC trace from raw predictions created in this cell, not model_meta or app code.
from scipy.stats import rankdata
ranks = rankdata(model_prob, method="average")
pos = y_test == 1; traced_auc = float((ranks[pos].sum() - pos.sum()*(pos.sum()+1)/2) / (pos.sum()*(~pos).sum()))
traced_fp = int(((model_prob >= THRESHOLD) & (y_test == 0)).sum())
perm = permutation_importance(chosen_model, train[FEATURES], train.productive, n_repeats=5, random_state=RANDOM_STATE, scoring="roc_auc")
importance = sorted([{"feature": f, "mean_auc_drop": float(v)} for f,v in zip(FEATURES, perm.importances_mean)], key=lambda x: x["mean_auc_drop"], reverse=True)
trip_type_response = {}
for trip_value, hours in [("inshore", 4), ("offshore", 8)]:
    probe = train[FEATURES].copy()
    probe["trip_type"] = trip_value
    probe["trip_hours"] = hours
    trip_type_response[trip_value] = float(chosen_model.predict_proba(probe)[:, 1].mean())
verification = {"leaky_auc": leaky_auc, "real_auc": test_metrics["auc"], "trailing_auc_without": no_trailing_auc, "trailing_auc_with": test_metrics["auc"], "random_split_auc": random_auc, "traced_auc": traced_auc, "traced_auc_matches": bool(np.isclose(traced_auc, test_metrics["auc"])), "traced_fp": traced_fp, "traced_fp_matches": bool(traced_fp == test_metrics["fp"]), "permutation_importance": importance, "trailing_rate_uses_prior_days": True, "partial_dependence_trip_type": trip_type_response}
Path("verification_results.json").write_text(json.dumps(verification, indent=2), encoding="utf-8")
print(json.dumps(verification, indent=2))